In [1]:
import sys, os

In [2]:
# notebook parameters, will be replaced by script arguments
application_name = "My Application"
s3a_access_key = os.environ.get("s3a_access_key")
s3a_secret_key = os.environ.get("s3a_secret_key")
input_file_1 = "s3a://uga-data-lake/poem.txt"
output_file_1 = "s3a://uga-data-lake/poem_wc1.txt"

In [4]:
import pyspark
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F
pyspark.__version__

'4.1.2'

In [5]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    application_name = sys.argv[1]
    s3a_access_key = sys.argv[2]
    s3a_secret_key = sys.argv[3]
    input_file_1 = sys.argv[4]
    output_file_1 = sys.argv[5]
    interactive = False
else:
    interactive = True
    print("Running as a notebook")
    pv = !{sys.executable} --version
    print("Python version ", pv)
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    print("PySpark version ", pyspark.__version__)
    jv = !java --version
    print("Java version ", jv)

Running as a notebook
Python version  ['Python 3.12.13']
PySpark version  4.1.2
Java version  ['openjdk 17.0.19 2026-04-21', 'OpenJDK Runtime Environment (build 17.0.19+10-1-deb12u2-Debian)', 'OpenJDK 64-Bit Server VM (build 17.0.19+10-1-deb12u2-Debian, mixed mode, sharing)']


In [6]:
def show(df):
    if interactive:
        df.show()

In [7]:
spark = (SparkSession.builder.appName(application_name)
         .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
         .config('spark.hadoop.fs.s3a.access.key', s3a_access_key)
         .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
         .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-2.amazonaws.com")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-26144e5f-10e6-4981-b305-88c1d4c9d106;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 121ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.analyt

In [8]:
input_data = spark.read.text(input_file_1)
show(input_data)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------------------+
|               value|
+--------------------+
|Two roads diverge...|
|And sorry I could...|
|And be one travel...|
|And looked down o...|
|To where it bent ...|
|                    |
|Then took the oth...|
|And having perhap...|
|Because it was gr...|
|Though as for tha...|
|Had worn them rea...|
|                    |
|And both that mor...|
|In leaves no step...|
|Oh, I kept the fi...|
|Yet knowing how w...|
|I doubted if I sh...|
|                    |
|I shall be tellin...|
|Somewhere ages an...|
+--------------------+
only showing top 20 rows


In [10]:
# split the lines into words
words = input_data.select(F.explode(F.split(input_data.value, r"\s+")).alias("word"))
show(words)

+---------+
|     word|
+---------+
|      Two|
|    roads|
| diverged|
|       in|
|        a|
|   yellow|
|    wood,|
|      And|
|    sorry|
|        I|
|    could|
|      not|
|   travel|
|     both|
|      And|
|       be|
|      one|
|traveler,|
|     long|
|        I|
+---------+
only showing top 20 rows


In [11]:
counts = words.groupBy("word").count().sort("count", ascending=False)
show(counts)

+--------+-----+
|    word|count|
+--------+-----+
|     the|    8|
|       I|    8|
|     And|    6|
|      as|    5|
|     one|    3|
|      in|    3|
|     and|    3|
|    that|    3|
|       a|    3|
|        |    3|
|   could|    2|
|     for|    2|
|      be|    2|
|    both|    2|
|      it|    2|
|    took|    2|
|   wood,|    2|
|     Two|    2|
|diverged|    2|
|    ages|    2|
+--------+-----+
only showing top 20 rows


In [12]:
counts.write.mode("overwrite").csv(output_file_1)

In [13]:
rdd = input_data.rdd
rdd.take(5)

[Row(value='Two roads diverged in a yellow wood,'),
 Row(value='And sorry I could not travel both'),
 Row(value='And be one traveler, long I stood'),
 Row(value='And looked down one as far as I could'),
 Row(value='To where it bent in the undergrowth;')]

In [14]:
rdd.map(lambda x: x.value.upper()).take(5)

['TWO ROADS DIVERGED IN A YELLOW WOOD,',
 'AND SORRY I COULD NOT TRAVEL BOTH',
 'AND BE ONE TRAVELER, LONG I STOOD',
 'AND LOOKED DOWN ONE AS FAR AS I COULD',
 'TO WHERE IT BENT IN THE UNDERGROWTH;']

In [15]:
spark.stop()